## Lab 5
In this lab, you will implement the random forest algorithm to build models for classification.

In [16]:
import numpy as np
import pandas as pd
import seaborn as sbs
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

Import the data in 'car.csv', split it into training, testing and validation sets.

In [17]:
car = pd.read_csv('car.csv')
car

,vhigh,vhigh.1,2,2.1,small,low,unacc
0,vhigh,vhigh,2,2,small,med,unacc
1,vhigh,vhigh,2,2,small,high,unacc
2,vhigh,vhigh,2,2,med,low,unacc
3,vhigh,vhigh,2,2,med,med,unacc
4,vhigh,vhigh,2,2,med,high,unacc
...,...,...,...,...,...,...,...
1722,low,low,5more,more,med,med,good
1723,low,low,5more,more,med,high,vgood
1724,low,low,5more,more,big,low,unacc
1725,low,low,5more,more,big,med,good


In [ ]:
def encode_column(col, mapping):
    return np.array([mapping[val] for val in col])

def load_car_dataset(path):
    # Define mappings
    buying_map = {'vhigh': 3, 'high': 2, 'med': 1, 'low': 0}
    maint_map = {'vhigh': 3, 'high': 2, 'med': 1, 'low': 0}
    doors_map = {'2': 2, '3': 3, '4': 4, '5more': 5}
    persons_map = {'2': 2, '4': 4, 'more': 5}
    lug_boot_map = {'small': 0, 'med': 1, 'big': 2}
    safety_map = {'low': 0, 'med': 1, 'high': 2}
    class_map = {'unacc': 0, 'acc': 1, 'good': 2, 'vgood': 3}

    raw = np.genfromtxt(path, delimiter=',', dtype=str)

    # Separate features and target variable
    buying     = encode_column(raw[:, 0], buying_map)
    maint      = encode_column(raw[:, 1], maint_map)
    doors      = encode_column(raw[:, 2], doors_map)
    persons    = encode_column(raw[:, 3], persons_map)
    lug_boot   = encode_column(raw[:, 4], lug_boot_map)
    safety     = encode_column(raw[:, 5], safety_map)
    y          = encode_column(raw[:, 6], class_map)

    # Stack X y reshapes
    X = np.column_stack((buying, maint, doors, persons, lug_boot, safety))
    y = y.reshape(-1, 1)

    return X, y

def split_data(X, y, train_ratio=0.6, val_ratio=0.2, seed=42):
    np.random.seed(seed)
    indices = np.random.permutation(len(X))
    n_train = int(len(X) * train_ratio)
    n_val = int(len(X) * val_ratio)

    train_idx = indices[:n_train]
    val_idx = indices[n_train:n_train+n_val]
    test_idx = indices[n_train+n_val:]

    return {
        'X_train': X[train_idx],
        'y_train': y[train_idx],
        'X_val': X[val_idx],
        'y_val': y[val_idx],
        'X_test': X[test_idx],
        'y_test': y[test_idx]
    }

X, y = load_car_dataset("car.csv") 
data = split_data(X, y)

Built the function you need to train a normal decision tree:

In [19]:
def gini(y):
    classes, counts = np.unique(y, return_counts=True)
    probs = counts / len(y)
    return 1 - np.sum(probs ** 2)

def best_split_classification(X, y, feature_idxs):
    m, n = X.shape
    best_feat, best_thresh, best_impurity = None, None, float('inf')

    for feature in feature_idxs:
        thresholds = np.unique(X[:, feature])
        for t in thresholds:
            left = X[:, feature] <= t
            right = X[:, feature] > t
            if left.sum() == 0 or right.sum() == 0:
                continue
            gini_left = gini(y[left])
            gini_right = gini(y[right])
            total_impurity = (left.sum() * gini_left + right.sum() * gini_right) / m
            if total_impurity < best_impurity:
                best_feat, best_thresh, best_impurity = feature, t, total_impurity
    return best_feat, best_thresh

def build_tree_classification(X, y, depth=0, max_depth=5, min_samples=5, max_features=None):
    if depth >= max_depth or len(y) <= min_samples:
        # Return the most common class label
        classes, counts = np.unique(y, return_counts=True)
        return classes[np.argmax(counts)]

    n_features = X.shape[1]
    feature_idxs = np.random.choice(n_features, size=max_features or n_features, replace=False)
    feat, thresh = best_split_classification(X, y, feature_idxs)

    if feat is None:
        classes, counts = np.unique(y, return_counts=True)
        return classes[np.argmax(counts)]

    left = X[:, feat] <= thresh
    right = X[:, feat] > thresh

    left_branch = build_tree_classification(X[left], y[left], depth + 1, max_depth, min_samples, max_features)
    right_branch = build_tree_classification(X[right], y[right], depth + 1, max_depth, min_samples, max_features)

    return (feat, thresh, left_branch, right_branch)


Buit a function that train a random forest:

In [20]:
def bootstrap_sample(X, y):
    n = len(y)
    idxs = np.random.choice(n, size=n, replace=True)
    return X[idxs], y[idxs]

def predict_one(x, tree):
    if not isinstance(tree, tuple):
        return tree
    feat, thresh, left, right = tree
    return predict_one(x, left) if x[feat] <= thresh else predict_one(x, right)

def predict_tree(X, tree):
    return np.array([predict_one(x, tree) for x in X])

def fit_random_forest_classification(X, y, n_trees=10, max_depth=5, min_samples=5, max_features=None):
    forest = []
    for _ in range(n_trees):
        X_sample, y_sample = bootstrap_sample(X, y)
        tree = build_tree_classification(X_sample, y_sample, max_depth=max_depth, min_samples=min_samples, max_features=max_features)
        forest.append(tree)
    return forest

def predict_random_forest_classification(X, forest):
    preds = np.array([predict_tree(X, tree) for tree in forest])
    # Votación mayoritaria (modo)
    from scipy.stats import mode
    return mode(preds, axis=0).mode[0]

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)



Built a function that predicts the class of a new point given your trained random forest

In [21]:
def predict_point_classification(x, forest):
    votes = np.array([predict_one(x, tree) for tree in forest])
    classes, counts = np.unique(votes, return_counts=True)
    return classes[np.argmax(counts)]

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def classification_report(y_true, y_pred):
    acc = accuracy(y_true, y_pred)
    return {"accuracy": acc}


Builld functions to evaluate your predictions, use this functions to find optimal hyperparameters using also our validation set.

In [22]:
def grid_search_classification(X_train, y_train, X_val, y_val, param_grid):
    best_params = None
    best_acc = -1

    for n_trees in param_grid['n_trees']:
        for max_depth in param_grid['max_depth']:
            for max_features in param_grid['max_features']:
                forest = fit_random_forest_classification(
                    X_train, y_train,
                    n_trees=n_trees,
                    max_depth=max_depth,
                    min_samples=5,
                    max_features=max_features
                )
                y_pred = predict_random_forest_classification(X_val, forest)
                acc = accuracy(y_val, y_pred)

                if acc > best_acc:
                    best_acc = acc
                    best_params = {
                        'n_trees': n_trees,
                        'max_depth': max_depth,
                        'max_features': max_features
                    }

    return best_params


Train a final random forest using your optimal hyperparameters and both the training and validation sets. Predict for the datapoints in the testing set and evaluate your final predictions.

In [23]:
def train_and_evaluate_final_classifier(data):
    # Combinate train and validation sets
    X_final = np.vstack((data['X_train'], data['X_val']))
    y_final = np.concatenate((data['y_train'], data['y_val']))

    # Definite hyperparameter grid
    param_grid = {
        'n_trees': [5, 10, 20],
        'max_depth': [3, 5, 7],
        'max_features': [2, 3, 4]
    }

    # Buscar mejores hiperparámetros usando val
    best_params = grid_search_classification(
        data['X_train'], data['y_train'].ravel(),
        data['X_val'], data['y_val'].ravel(),
        param_grid
    )

    print("Mejores hiperparámetros:", best_params)

    # Training final classifier with best parameters
    forest = fit_random_forest_classification(
        X_final, y_final.ravel(),
        n_trees=best_params['n_trees'],
        max_depth=best_params['max_depth'],
        min_samples=5,
        max_features=best_params['max_features']
    )

    # Evaluating on test set
    y_test = data['y_test'].ravel()
    y_pred = predict_random_forest_classification(data['X_test'], forest)

    print("Evaluación en test:", classification_report(y_test, y_pred))

    return forest


In [24]:
X, y = load_car_dataset("car.csv")
data = split_data(X, y)

forest = train_and_evaluate_final_classifier(data)


Mejores hiperparámetros: {'n_trees': 5, 'max_depth': 3, 'max_features': 2}
Evaluación en test: {'accuracy': np.float64(0.6570605187319885)}
